# Chapter 8 - Data Modification: Propositions & Queries
**Name:** Andrew Laboy

**Group Number:** EOS_grp_ 1 

**Date:** 04-12-26  

**Database:** Northwinds2024Student  

---


## Proposition 1
**Problem Statement:** Create a demo table and insert sample employee bonus records using INSERT,INTO and VALUES.

**Why This Query Is Special:** It shows the a basic form of inserting multiple rows with a single INSERT statement.

In [1]:

USE Northwinds2024Student;
GO

DROP TABLE IF EXISTS dbo.EmployeeBonusDemo;

CREATE TABLE dbo.EmployeeBonusDemo
(
    BonusID INT IDENTITY(1,1) PRIMARY KEY,
    EmpName NVARCHAR(50) NOT NULL,
    BonusAmount MONEY NOT NULL,
    BonusDate DATE NOT NULL
);

INSERT INTO dbo.EmployeeBonusDemo (EmpName, BonusAmount, BonusDate)
VALUES
    ('Sara', 500.00, '2026-01-15'),
    ('James', 750.00, '2026-01-15'),
    ('Maria', 600.00, '2026-02-01');

SELECT * FROM dbo.EmployeeBonusDemo;

Error: No connection selected.

---
## Proposition 2
**Problem Statement:** Copy all products that cost more than $30 into a separate demo table using INSERT INTO and SELECT.

**Why This Query Is Special:** It shows how to populate a new table using data from an existing table with a WHERE filter.

In [2]:

USE Northwinds2024Student;
GO

DROP TABLE IF EXISTS dbo.ExpensiveProductsDemo;

CREATE TABLE dbo.ExpensiveProductsDemo
(
    ProductId INT PRIMARY KEY,
    ProductName NVARCHAR(MAX),
    UnitPrice MONEY
);

INSERT INTO dbo.ExpensiveProductsDemo (ProductId, ProductName, UnitPrice)
SELECT p.ProductId, p.ProductName, p.UnitPrice
FROM Production.Product AS p
WHERE p.UnitPrice > 30.00;

SELECT * FROM dbo.ExpensiveProductsDemo
ORDER BY UnitPrice DESC;

Error: No connection selected.

---
## Proposition 3
**Problem Statement:** Update the price of all products in a demo table that belong to the Beverages category by increasing them by 10%.

**Why This Query Is Special:** It uses UPDATE with a simple JOIN to connect two tables and applies a business rule.

In [3]:

USE Northwinds2024Student;
GO

DROP TABLE IF EXISTS dbo.ProductPriceDemo;
GO

SELECT p.ProductId, p.ProductName, p.UnitPrice, c.CategoryName
INTO dbo.ProductPriceDemo
FROM Production.Product AS p
INNER JOIN Production.Category AS c ON p.CategoryId = c.CategoryId;

UPDATE dbo.ProductPriceDemo
SET UnitPrice = UnitPrice * 1.10
WHERE CategoryName = N'Beverages';

SELECT * FROM dbo.ProductPriceDemo
WHERE CategoryName = N'Beverages'
ORDER BY UnitPrice DESC;

Commands completed successfully.

Commands completed successfully.

(77 rows affected)
(12 rows affected)
(12 rows affected)

ProductId | ProductName   | UnitPrice | CategoryName
----------+---------------+-----------+-------------
38        | Product QDOMO | 289.85    | Beverages   
43        | Product ZZZHR | 50.60     | Beverages   
2         | Product RECZE | 20.90     | Beverages   
1         | Product HHYDP | 19.80     | Beverages   
39        | Product LSOFL | 19.80     | Beverages   
35        | Product NEVTJ | 19.80     | Beverages   
76        | Product JYGFE | 19.80     | Beverages   
70        | Product TOONT | 16.50     | Beverages   
67        | Product XLXQF | 15.40     | Beverages   
34        | Product SWNJY | 15.40     | Beverages   
75        | Product BWRLG | 8.525     | Beverages   
24        | Product QOGNU | 4.95      | Beverages   
(12 rows)

---
## Proposition 4
**Problem Statement:** Delete all orders from a demo table where the freight charge was zero.

**Why This Query Is Special:** It shows a straightforward DELETE with a WHERE clause to remove rows that meet a specific condition.

In [4]:

USE Northwinds2024Student;
GO

DROP TABLE IF EXISTS dbo.OrderFreightDemo;
GO

SELECT o.OrderId, o.CustomerId, o.OrderDate, o.Freight
INTO dbo.OrderFreightDemo
FROM Sales.[Order] AS o;

SELECT COUNT(*) AS before_count FROM dbo.OrderFreightDemo;

DELETE FROM dbo.OrderFreightDemo
WHERE Freight = 0;

SELECT COUNT(*) AS after_count FROM dbo.OrderFreightDemo;

Commands completed successfully.

Commands completed successfully.

(830 rows affected)
(1 row affected)
(0 rows affected)
(1 row affected)

before_count
------------
830         
(1 row)

after_count
-----------
830        
(1 row)

---
## Proposition 5
**Problem Statement:** Create a snapshot of all employees and their city using SELECT INTO.

**Why This Query Is Special:** SELECT INTO creates the table and fills it in one statement, which is simpler than creating the table first and then inserting.

In [5]:

USE Northwinds2024Student;
GO

DROP TABLE IF EXISTS dbo.EmployeeCitySnapshot;
GO

SELECT e.EmployeeId,
       e.EmployeeFirstName,
       e.EmployeeLastName,
       e.EmployeeCity,
       e.EmployeeCountry
INTO dbo.EmployeeCitySnapshot
FROM HumanResources.Employee AS e;

SELECT * FROM dbo.EmployeeCitySnapshot
ORDER BY EmployeeCountry, EmployeeCity;

Commands completed successfully.

Commands completed successfully.

(9 rows affected)
(9 rows affected)

EmployeeId | EmployeeFirstName | EmployeeLastName | EmployeeCity | EmployeeCountry
-----------+-------------------+------------------+--------------+----------------
5          | Sven              | Mortensen        | London       | UK             
6          | Paul              | Suurs            | London       | UK             
7          | Russell           | King             | London       | UK             
9          | Patricia          | Doyle            | London       | UK             
3          | Judy              | Lew              | Kirkland     | USA            
4          | Yael              | Peled            | Redmond      | USA            
1          | Sara              | Davis            | Seattle      | USA            
8          | Maria             | Cameron          | Seattle      | USA            
2          | Don               | Funk             | Tacoma       | USA            
(9 rows)

---
## Proposition 6
**Problem Statement:** Use TRUNCATE TABLE to quickly empty a demo table after it is no longer needed.

**Why This Query Is Special:** It shows the difference between DELETE  and TRUNCATE, which is important for performance.

In [6]:

USE Northwinds2024Student;
GO

SELECT COUNT(*) AS before_count FROM dbo.EmployeeCitySnapshot;

TRUNCATE TABLE dbo.EmployeeCitySnapshot;

SELECT COUNT(*) AS after_count FROM dbo.EmployeeCitySnapshot;

Commands completed successfully.

(1 row affected)
(1 row affected)

before_count
------------
9           
(1 row)

after_count
-----------
0          
(1 row)

---
## Proposition 7
**Problem Statement:** Use UPDATE with a CASE expression to label orders as High, Medium, or Low based on their freight cost.

**Why This Query Is Special:** It uses conditional logic inside UPDATE to classify rows into categories instead of just setting one value.

In [7]:

USE Northwinds2024Student;
GO

DROP TABLE IF EXISTS dbo.OrderFreightLabelDemo;
GO

SELECT o.OrderId,
       o.CustomerId,
       o.Freight,
       CAST(NULL AS VARCHAR(10)) AS FreightLabel
INTO dbo.OrderFreightLabelDemo
FROM Sales.[Order] AS o;

UPDATE dbo.OrderFreightLabelDemo
SET FreightLabel = CASE
                       WHEN Freight >= 100 THEN 'High'
                       WHEN Freight >= 30 THEN 'Medium'
                       ELSE 'Low'
                   END;

SELECT FreightLabel, COUNT(*) AS OrderCount
FROM dbo.OrderFreightLabelDemo
GROUP BY FreightLabel;

Commands completed successfully.

Commands completed successfully.

(830 rows affected)
(830 rows affected)
(3 rows affected)

FreightLabel | OrderCount
-------------+-----------
High         | 187       
Low          | 347       
Medium       | 296       
(3 rows)

---
## Proposition 8
**Problem Statement:** Use the OUTPUT clause during an UPDATE to capture which product prices changed and what the old and new values were.

**Why This Query Is Special:** OUTPUT lets you see exactly what changed during a modification, which is useful for tracking and auditing.

In [8]:

USE Northwinds2024Student;
GO

DROP TABLE IF EXISTS dbo.PriceChangeDemo;
GO

SELECT p.ProductId, p.ProductName, p.UnitPrice
INTO dbo.PriceChangeDemo
FROM Production.Product AS p
WHERE p.CategoryId = 1;

UPDATE dbo.PriceChangeDemo
SET UnitPrice = UnitPrice * 1.05
OUTPUT deleted.ProductId,
       deleted.ProductName,
       deleted.UnitPrice AS OldPrice,
       inserted.UnitPrice AS NewPrice;

Commands completed successfully.

Commands completed successfully.

(12 rows affected)
(12 rows affected)

ProductId | ProductName   | OldPrice | NewPrice
----------+---------------+----------+---------
1         | Product HHYDP | 18.00    | 18.90   
2         | Product RECZE | 19.00    | 19.95   
24        | Product QOGNU | 4.50     | 4.725   
34        | Product SWNJY | 14.00    | 14.70   
35        | Product NEVTJ | 18.00    | 18.90   
38        | Product QDOMO | 263.50   | 276.675 
39        | Product LSOFL | 18.00    | 18.90   
43        | Product ZZZHR | 46.00    | 48.30   
67        | Product XLXQF | 14.00    | 14.70   
70        | Product TOONT | 15.00    | 15.75   
75        | Product BWRLG | 7.75     | 8.1375  
76        | Product JYGFE | 18.00    | 18.90   
(12 rows)

---
## Proposition 9
**Problem Statement:** Delete the 5 cheapest products from a demo table using DELETE with a subquery.

**Why This Query Is Special:** It combines DELETE with a subquery to target specific rows based on price, rather than just a simple WHERE condition.

In [9]:

USE Northwinds2024Student;
GO

DROP TABLE IF EXISTS dbo.ProductCleanupDemo;
GO

SELECT p.ProductId, p.ProductName, p.UnitPrice
INTO dbo.ProductCleanupDemo
FROM Production.Product AS p;

SELECT COUNT(*) AS before_count FROM dbo.ProductCleanupDemo;

DELETE FROM dbo.ProductCleanupDemo
WHERE ProductId IN
(
    SELECT TOP 5 ProductId
    FROM dbo.ProductCleanupDemo
    ORDER BY UnitPrice ASC
);

SELECT COUNT(*) AS after_count FROM dbo.ProductCleanupDemo;

Commands completed successfully.

Commands completed successfully.

(77 rows affected)
(1 row affected)
(5 rows affected)
(1 row affected)

before_count
------------
77          
(1 row)

after_count
-----------
72         
(1 row)

---
## Proposition 10
**Problem Statement:** Use MERGE to synchronize a customer list demo table and update existing customer cities and insert any customers that are missing.

**Why This Query Is Special:** MERGE handles both INSERT and UPDATE in one statement, which is used for when you need to keep a table in sync with a source.

In [10]:

USE Northwinds2024Student;
GO

DROP TABLE IF EXISTS dbo.CustomerSyncDemo;
GO

SELECT TOP 45
       CAST(c.CustomerId AS INT) AS CustomerId,
       c.CustomerCompanyName,
       c.CustomerCity
INTO dbo.CustomerSyncDemo
FROM Sales.Customer AS c
ORDER BY c.CustomerId;

UPDATE dbo.CustomerSyncDemo
SET CustomerCity = N'Unknown'
WHERE CustomerId <= 10;

MERGE dbo.CustomerSyncDemo AS tgt
USING
(
    SELECT CAST(c.CustomerId AS INT) AS CustomerId,
           c.CustomerCompanyName,
           c.CustomerCity
    FROM Sales.Customer AS c
) AS src
ON tgt.CustomerId = src.CustomerId
WHEN MATCHED THEN
    UPDATE SET CustomerCity = src.CustomerCity,
               CustomerCompanyName = src.CustomerCompanyName
WHEN NOT MATCHED BY TARGET THEN
    INSERT (CustomerId, CustomerCompanyName, CustomerCity)
    VALUES (src.CustomerId, src.CustomerCompanyName, src.CustomerCity);

SELECT * FROM dbo.CustomerSyncDemo
ORDER BY CustomerId;

Commands completed successfully.

Commands completed successfully.

(45 rows affected)
(10 rows affected)
(91 rows affected)
(91 rows affected)

CustomerId | CustomerCompanyName | CustomerCity   
-----------+---------------------+----------------
1          | Customer NRZBB      | Berlin         
2          | Customer MLTDN      | México D.F.    
3          | Customer KBUDE      | México D.F.    
4          | Customer HFBZG      | London         
5          | Customer HGVLZ      | Luleå          
6          | Customer XHXJV      | Mannheim       
7          | Customer QXVLA      | Strasbourg     
8          | Customer QUHWH      | Madrid         
9          | Customer RTXGC      | Marseille      
10         | Customer EEALV      | Tsawassen      
11         | Customer UBHAU      | London         
12         | Customer PSNMQ      | Buenos Aires   
13         | Customer VMLOG      | México D.F.    
14         | Customer WNMAF      | Bern           
15         | Customer JUWXK      | Sao Paulo      
16         | Customer GYBBY      | London         
17         | Customer FEVNN      | Aachen         
18         | Customer BSVAR    

---
## NACE Competencies Reflection
- **Equity & Inclusion:** I made sure every group memeber had an equal opprtunity to contribute by constantly asking for updates and providing different templates to help make things go much more organized. 
- **Career & Self-Development:** I was able to get more practice using INSERT, UPDATE, DELETED, MERGE and OUTPUT and applying it to real database scenarios. 
- **Communication:** For each proposition I provided both english and code so people who dont understand the code can still understand what the code is aiming for.
- **Critical Thinking:** I had to make sure each query was actually solving what the proposition was asking, especially for when I used JOIN, subqueries and CASE expressions. 
- **Professionalism:** I made sure to name the file correctly, submit on time and follow the notebook structure that was assigned.
- **Technology:** I used Visual Studio and SQL Server Express to write and test all my queries using the Northwinds2024Student database.
- **Leadership:** I took the role as homework leader and setup both the group chat with organized channels and templates as well as created the GitHub repo, and wrote paragraphs in each channel explaining everyones tasks that they should follow.
- **Teamwork:** Unfortunatly, I was only able to collaborate with Amrina as they were the only person who was willing to try and do what they were assigned.